# 11 - Cross-Asset and Macroeconomic Feature Engineering

**Project:** AI Quantitative Trading & Strategy Optimisation System  
**Phase:** 6 — Feature Engineering and Market Regime Detection  
**Status:** Executed against the recorded local market acquisitions and current dataset configurations on 14 November 2026; owner review is pending.

## 1. Objective and scope

Investigate whether information from the approved market universe and UK macroeconomic datasets can be represented as candidate features beyond the target asset’s own history. Cross-asset transformations use close-to-close returns and preserve each instrument’s calendar. Macroeconomic candidates are inventoried with their native frequency and point-in-time restrictions.

This notebook does not train a model, establish incremental predictive value, or implement a strategy. It does not join macroeconomic observations to daily decisions because the available datasets do not contain validated historical release timestamps and vintages for every observation.

## 2. Research hypotheses

- The FTSE 100 return may represent broad UK equity-market conditions beyond an individual utility’s own history.
- SSE.L and SVT.L returns may capture sector-specific information shared with another UK utility.
- Bank Rate, SONIA and gilt yields may represent monetary and financing conditions; the 10-year minus 2-year gilt spread is a candidate transformation.
- CPI, Core CPI and CPIH may represent inflation pressures relevant to utility costs and cash flows.
- GDP growth may represent broader demand conditions.

These are hypotheses for future evaluation. Correlation or economic plausibility alone does not establish predictive value. CPI measures are related; CPIH–CPI and CPI–Core differences are not independent evidence. Bank Rate and SONIA are distinct measures, and SONIA has a methodology change effective 23 April 2018.

## 3. Market-feature temporal convention

Daily market prices are indexed by observation date. Same-date close-to-close returns are aligned by exact date on the union of observed calendars. Missing sessions remain missing; no forward-fill, interpolation, resampling or synthetic weekend values are created. The feature at date *t* uses close information through *t* and is only available after the relevant market close. It must not imply same-session execution; subsequent forecasting work must define a decision after close and execution no earlier than the next eligible session.

The market source files are historical Yahoo Finance acquisitions. Their provider, licensing, revision and timestamp limitations remain. The Phase 3 equity table did not assign formal PIT labels, so this notebook does not infer one. Same-date alignment establishes date matching, not independently verified publication timestamps or strict PIT eligibility.

## 4. Macroeconomic point-in-time gate

Observation date/period, publication or availability timestamp, acquisition timestamp, vintage, decision timestamp and execution timestamp are distinct fields. The existing macro readers expose observation dates and dataset-level acquisition provenance, but do not establish a validated per-observation historical availability timestamp and vintage for every series. The dataset configurations also do not require availability timestamps. Therefore, these datasets remain candidate research inputs and are not joined to historical daily decisions here.

The approved conservative Bank of England database-series availability policy is 10:00 on the next London business day, but this is not a reconstructed complete release calendar. Bank Rate and SONIA remain `PIT_CONDITIONAL`; gilts and GDP remain `PIT_UNVALIDATED`. Inflation is conditional, with incomplete historical release timestamps and vintages. No observation date is treated as a publication date. The helper in `src/features/cross_asset.py` fails closed if either availability or acquisition timestamps are missing, timezone-naive, or after the decision timestamp; passing it would still not prove the underlying source policy is valid.

## 5. Candidate feature register

| Candidate | Transformation | Native frequency | Hypothesis | PIT status and use restriction |
|---|---|---|---|---|
| FTSE 100 | 1-, 5- and 20-session close returns | Daily | Broad UK equity conditions | Historical market timestamp/revision limits; date alignment only, no strict PIT claim |
| SSE.L, SVT.L | 1-, 5- and 20-session close returns | Daily | Utility-sector co-movement | Same limits as other Yahoo market data; not a causal claim |
| Bank Rate | Level and change between observations | Business daily | Monetary policy and financing conditions | `PIT_CONDITIONAL`; next-London-business-day conservative policy; validate row-level release eligibility before use |
| SONIA | Level and change between observations | Business daily | Overnight funding conditions | `PIT_CONDITIONAL`; availability policy and 23 April 2018 methodology break must be considered |
| Gilt 2Y, 10Y | Levels and 10Y–2Y spread | Business daily | Short/long financing and policy expectations | `PIT_UNVALIDATED`; not eligible for strict PIT backtesting |
| CPI, Core CPI, CPIH | Published annual-rate level and change between monthly observations | Monthly | Inflation pressures | `PIT_CONDITIONAL`; publication delays/vintages incomplete; no daily join |
| Real GDP, GDP growth | Level/growth and changes at native frequency | Quarterly/annual | Economic activity and demand | `PIT_UNVALIDATED`; revisions and release timing unresolved |

The current configurations mark commercial use and redistribution approval as pending. Data-source access is not a licence grant. Energy prices and FX are deferred because no corresponding approved dataset configuration is present in the inspected repository.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs" / "assets").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.cross_asset import build_cross_asset_return_panel

ASSET_CONFIG_PATHS = sorted((PROJECT_ROOT / "configs" / "assets").glob("*.yaml"))
assert ASSET_CONFIG_PATHS, "No configured market assets were found."
print(f"Repository: {PROJECT_ROOT}")
print(f"Configured market assets: {len(ASSET_CONFIG_PATHS)}")

Repository: /home/anderjcruz/projects/Quantitative-Trading-Strategy
Configured market assets: 4


## 6. Load configured market observations

Asset symbols are taken from the asset YAML files. Exactly one acquired Parquet file per configured asset is required; ambiguous acquisition versions stop the notebook for explicit review. Files are opened read-only. `Close` is used consistently with Notebook 10; denomination conversion is not applied.

In [2]:
market_prices = {}
market_source_inventory = []

for config_path in ASSET_CONFIG_PATHS:
    with config_path.open(encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)
    symbol = config["provider"]["symbol"]
    acquisition_files = sorted(
        (PROJECT_ROOT / "data" / "raw" / "market" / "yahoo_finance" / symbol)
        .glob(f"acquisition_*/{symbol}.parquet")
    )
    if len(acquisition_files) != 1:
        raise RuntimeError(
            f"Expected one recorded acquisition for {symbol}; found {len(acquisition_files)}."
        )

    frame = pd.read_parquet(acquisition_files[0])
    if "Date" in frame.columns:
        frame = frame.set_index("Date")
    frame.index = pd.to_datetime(frame.index)
    frame.index.name = "observation_date"
    if not frame.index.is_monotonic_increasing or frame.index.has_duplicates:
        raise ValueError(f"{symbol} dates must be unique and chronological.")
    if "Close" not in frame:
        raise ValueError(f"The acquired series for {symbol} has no Close field.")

    market_prices[symbol] = frame["Close"].rename(symbol)
    market_source_inventory.append({
        "symbol": symbol,
        "name": config["asset"]["name"],
        "instrument_type": config["asset"]["instrument_type"],
        "price_unit": config["asset"]["price_unit"],
        "unit_validation_status": config["asset"]["price_unit_validation_status"],
        "acquisition_file": acquisition_files[0].relative_to(PROJECT_ROOT).as_posix(),
        "observations": len(frame),
        "first_observation": frame.index.min(),
        "last_observation": frame.index.max(),
    })

market_source_inventory = pd.DataFrame(market_source_inventory).set_index("symbol")
market_source_inventory

,name,instrument_type,price_unit,unit_validation_status,acquisition_file,observations,first_observation,last_observation
symbol,,,,,,,,
^FTSE,FTSE 100 Index,index,index_points,not_applicable,data/raw/market/yahoo_finance/^FTSE/acquisitio...,11004,1984-01-03,2026-11-02
NG.L,National Grid plc,equity,GBp,pending,data/raw/market/yahoo_finance/NG.L/acquisition...,7890,1995-12-11,2026-11-02
SSE.L,SSE plc,equity,GBp,pending,data/raw/market/yahoo_finance/SSE.L/acquisitio...,9060,1991-06-17,2026-11-02
SVT.L,Severn Trent plc,equity,GBp,pending,data/raw/market/yahoo_finance/SVT.L/acquisitio...,9455,1989-12-11,2026-11-02


## 7. Build and inspect exact-date cross-asset returns

The return panel uses a union of exact observation dates. The resulting missing values make calendar differences visible instead of carrying old prices into later sessions. Returns are dimensionless, so NG.L, SSE.L and SVT.L remain in their provider quote units and FTSE remains in index points.

In [3]:
cross_asset_returns = build_cross_asset_return_panel(
    market_prices,
    lookbacks=(1, 5, 20),
)

coverage = pd.DataFrame({
    "non_missing_observations": cross_asset_returns.notna().sum(),
    "first_available_date": cross_asset_returns.apply(lambda column: column.first_valid_index()),
    "last_available_date": cross_asset_returns.apply(lambda column: column.last_valid_index()),
    "missing_observations": cross_asset_returns.isna().sum(),
})
coverage

,non_missing_observations,first_available_date,last_available_date,missing_observations
^FTSE__return_1,10684,1984-01-04,2026-10-30,320
^FTSE__return_5,10655,1984-01-10,2026-10-30,349
^FTSE__return_20,10631,1984-01-31,2026-10-30,373
NG.L__return_1,7888,1995-12-12,2026-10-30,3116
NG.L__return_5,7884,1995-12-18,2026-10-30,3120
NG.L__return_20,7869,1996-01-08,2026-10-30,3135
SSE.L__return_1,9058,1991-06-18,2026-10-30,1946
SSE.L__return_5,9054,1991-06-24,2026-10-30,1950
SSE.L__return_20,9039,1991-07-15,2026-10-30,1965
SVT.L__return_1,9453,1989-12-12,2026-10-30,1551


In [4]:
# A date-level view demonstrates exact calendar alignment; missing values remain explicit.
cross_asset_returns.tail(10)

,^FTSE__return_1,^FTSE__return_5,^FTSE__return_20,NG.L__return_1,NG.L__return_5,NG.L__return_20,SSE.L__return_1,SSE.L__return_5,SSE.L__return_20,SVT.L__return_1,SVT.L__return_5,SVT.L__return_20
observation_date,,,,,,,,,,,,
2026-10-20,-0.002945,0.007765,0.002307,0.017654,0.027153,0.052151,0.019561,0.015911,0.034427,0.014548,0.023612,0.032175
2026-10-21,0.001854,0.002611,0.004446,-0.002065,0.018979,0.064786,0.003132,0.016263,0.037247,0.001247,0.029487,0.055191
2026-10-22,-0.002595,-0.010645,0.004213,0.013245,0.025555,0.077939,0.000781,0.009449,0.035960,0.009340,0.008085,0.058785
2026-10-23,-0.003226,-0.017598,-0.000458,-0.002451,0.002875,0.083407,-0.016381,-0.020202,0.026873,-0.001234,0.005590,0.057479
2026-10-26,0.007193,0.000241,0.007712,0.001229,0.027743,0.084257,-0.014671,-0.007984,0.010985,0.003706,0.027830,0.058632
2026-10-27,-0.007504,-0.004332,0.004682,-0.001227,0.008674,0.087266,0.002012,-0.025059,0.021748,-0.005538,0.007481,0.053455
2026-10-28,0.000749,-0.005431,0.008344,0.013514,0.024421,0.080314,0.045783,0.016393,0.045363,0.006807,0.013076,0.051034
2026-10-29,0.002581,-0.000270,0.028173,0.013737,0.024918,0.106261,0.016513,0.032371,0.070360,0.006761,0.010487,0.065712
2026-10-30,0.000718,0.003685,0.025597,0.000000,0.027437,0.094200,-0.007556,0.041634,0.073559,0.009158,0.021001,0.060295


## 8. Macroeconomic source and eligibility inventory

This inventory reads only configuration metadata. It does not transform or join macroeconomic observations. The previous analyses provide economic context, while the configurations and Phase 3 classifications determine the limits on historical use.

In [5]:
DATASET_CONFIG_PATHS = sorted((PROJECT_ROOT / "configs" / "datasets").rglob("*.yaml"))
macro_status = {
    "UK_BANK_RATE": ("PIT_CONDITIONAL", "10:00 next London business day conservative policy; row-level release validation outstanding"),
    "UK_SONIA": ("PIT_CONDITIONAL", "10:00 next London business day conservative policy; methodology changed 23 April 2018"),
    "UK_GILT_NOMINAL_2Y": ("PIT_UNVALIDATED", "Historical availability and vintages unverified"),
    "UK_GILT_NOMINAL_10Y": ("PIT_UNVALIDATED", "Historical availability and vintages unverified"),
    "UK_CPI": ("PIT_CONDITIONAL", "Historical publication timestamps and vintages incomplete"),
    "UK_CORE_CPI": ("PIT_CONDITIONAL", "Historical publication timestamps and vintages incomplete"),
    "UK_CPIH": ("PIT_CONDITIONAL", "Historical publication timestamps and vintages incomplete"),
    "UK_REAL_GDP": ("PIT_UNVALIDATED", "Historical release timestamps and vintages not fully reconstructed"),
    "UK_GDP_GROWTH": ("PIT_UNVALIDATED", "Historical release timestamps and vintages not fully reconstructed"),
}

macro_inventory = []
for config_path in DATASET_CONFIG_PATHS:
    with config_path.open(encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)
    dataset = config.get("dataset", {})
    temporal = config.get("temporal", {})
    licensing = config.get("licensing", {})
    provider = config.get("provider", {})
    identifiers = dataset.get("dataset_ids", [dataset.get("dataset_id")])
    identifiers = [identifier for identifier in identifiers if identifier]
    for identifier in identifiers:
        pit_status, temporal_limit = macro_status.get(
            identifier, ("NOT_CLASSIFIED", "Review the existing project classification before use")
        )
        macro_inventory.append({
            "dataset_id": identifier,
            "dataset": dataset.get("dataset_name"),
            "provider": provider.get("name"),
            "frequency": config.get("data", {}).get("frequency", "see series configuration"),
            "unit": config.get("data", {}).get("unit", "see series configuration"),
            "publication_date_required": temporal.get("publication_date_required"),
            "availability_timestamp_required": temporal.get("availability_timestamp_required"),
            "pit_status": pit_status,
            "feature_join_eligible_now": False,
            "temporal_limit": temporal_limit,
            "licensing_status": licensing.get("status", "not recorded"),
        })

macro_inventory = pd.DataFrame(macro_inventory).set_index("dataset_id")
macro_inventory

,dataset,provider,frequency,unit,publication_date_required,availability_timestamp_required,pit_status,feature_join_eligible_now,temporal_limit,licensing_status
dataset_id,,,,,,,,,,
UK_BANK_RATE,Official Bank Rate,bank_of_england,business_daily,percent_per_annum,False,False,PIT_CONDITIONAL,False,10:00 next London business day conservative po...,pending_verification
UK_GILT_NOMINAL_10Y,UK Nominal Government Spot Yield — 10 Year,bank_of_england,business_daily,percent_per_annum,False,False,PIT_UNVALIDATED,False,Historical availability and vintages unverified,pending_verification
UK_GILT_NOMINAL_2Y,UK Nominal Government Spot Yield — 2 Year,bank_of_england,business_daily,percent_per_annum,False,False,PIT_UNVALIDATED,False,Historical availability and vintages unverified,pending_verification
UK_SONIA,Sterling Overnight Index Average,bank_of_england,business_daily,percent_per_annum,False,False,PIT_CONDITIONAL,False,10:00 next London business day conservative po...,pending_verification
UK_REAL_GDP,UK Quarterly National Accounts,office_for_national_statistics,see series configuration,see series configuration,False,False,PIT_UNVALIDATED,False,Historical release timestamps and vintages not...,pending_verification
UK_GDP_GROWTH,UK Quarterly National Accounts,office_for_national_statistics,see series configuration,see series configuration,False,False,PIT_UNVALIDATED,False,Historical release timestamps and vintages not...,pending_verification
UK_CPI,UK Consumer Price Inflation,office_for_national_statistics,see series configuration,see series configuration,False,False,PIT_CONDITIONAL,False,Historical publication timestamps and vintages...,pending_verification
UK_CORE_CPI,UK Consumer Price Inflation,office_for_national_statistics,see series configuration,see series configuration,False,False,PIT_CONDITIONAL,False,Historical publication timestamps and vintages...,pending_verification
UK_CPIH,UK Consumer Price Inflation,office_for_national_statistics,see series configuration,see series configuration,False,False,PIT_CONDITIONAL,False,Historical publication timestamps and vintages...,pending_verification


In [6]:
# Fail-closed example on an intentionally incomplete schema: no real dataset is passed here.
# Real macro joins are not attempted because the source rows lack validated availability timestamps.
assert not macro_inventory["feature_join_eligible_now"].any()
assert macro_inventory["availability_timestamp_required"].eq(False).all()
print("No configured macro series is eligible for a historical daily join in this notebook.")

No configured macro series is eligible for a historical daily join in this notebook.


## 9. Temporal validation helper

`point_in_time_eligibility` is available for later inputs that contain explicit, timezone-aware `availability_timestamp` and `acquisition_timestamp` fields alongside decision timestamps. It checks that both timestamps are no later than each decision and returns `False` for missing or timezone-naive values. It does not invent timestamps, establish source truth, reconstruct vintages, or override a dataset’s PIT classification. No actual macroeconomic series is passed through the helper here because the required row-level evidence is absent.

## 10. Findings and limitations

- Daily return transformations are implemented for the configured market universe; date alignment uses an outer exact-index join and exposes missing sessions.
- The panel’s same-date observations are close-to-close inputs available only after market close. Later use must not assume same-bar execution.
- Cross-asset return correlations, redundancy and incremental predictive value have not been assessed here.
- Macro candidate transformations remain levels, changes and the 10Y–2Y spread at native frequency. They are not aligned to daily decisions in the absence of validated availability timestamps and vintages.
- `PIT_CONDITIONAL` is not equivalent to a blanket approval for every historical row. `PIT_UNVALIDATED` series remain excluded from strict PIT research.
- Licensing remains pending at dataset level; Yahoo Finance remains exploratory research data.
- No FRED/ALFRED observations are downloaded or used.
- The notebook executed on 14 November 2026; outputs are stored here. The market return panel was calculated from local Raw acquisitions. Macro rows were not joined to decision dates because validated per-observation availability timestamps and vintages are not available. The reusable alignment and PIT checks were covered by deterministic unit tests.
